In [ ]:
import re

In [ ]:
def format_reward_fun(completions, **kwargs):
  Pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"

  rewards = []
  for completion in completions:
    match = re.search(Pattern, completion, re.DOTALL)
    if match:
      think_content = match.group(1).strip()
      answer_content = match.group(2).strip()

      if len(think_content) > 20 and len(answer_content) > 0:
        rewards.append(1.0)
      else:
        rewards.append(0.5)
    else:
      rewards.append(0.0)

  return rewards

sample_completion =  ["<think> I need to add 2 and 2. </think> <answer> 4 </answer>"]
print(f"📊 Format Reward Score: {format_reward_fun(sample_completion)[0]}")
print("✅ Format is correct!")

📊 Format Reward Score: 1.0
✅ Format is correct!


In [ ]:
print("\n" + "="*60)
print("📝 EXERCISE FOR YOU:")
print("="*60)
print("Write a reward function that checks BOTH format AND")
print("whether the final answer (inside <answer> tags) matches")
print("the correct answer from the dataset.")
print("\n💡 Hint: Add an 'answers' parameter and use zip()")


📝 EXERCISE FOR YOU:
Write a reward function that checks BOTH format AND
whether the final answer (inside <answer> tags) matches
the correct answer from the dataset.

💡 Hint: Add an 'answers' parameter and use zip()


In [ ]:
def format_and_correctness_reward_func(completions, answers, **kwargs):
  pattern =  r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
  rewards = []

  for completion, correct_answer in zip(completions, answers):
    match = re.search(pattern, completion, re.DOTALL)

    if match:
      think_content = match.group(1).strip()
      answer_content = match.group(2).strip()

      format_ok = len(think_content) > 20 and len(answer_content) > 0

      answer_clean = answer_content.strip().lower()
      correct_clean = str(correct_answer).strip().lower()

      is_correct = (answer_clean == correct_clean)

      if format_ok and is_correct:
        reward = 1.0
      elif format_ok and not is_correct:
        reward = 0.5
      elif not format_ok and is_correct:
        reward = 0.3
      else:
        reward = 0.0
      rewards.append(reward)
    else:
      rewards.append(0.0)

  return rewards

In [ ]:
print("\n" + "="*60)
print("🧪 TESTING THE COMBINED REWARD FUNCTION:")
print("="*60)


🧪 TESTING THE COMBINED REWARD FUNCTION:


In [ ]:
test_completions = [
    "<think> I need to add 2 and 2 together. </think> <answer> 4 </answer>",
    "<think> Adding 2 and 2 gives us 4. </think> <answer> 5 </answer>",
    "<think> I need to add these numbers. </think> <answer> 4 </answer>",
    "<answer> 4 </answer>",
    "4"
]

test_answer = [4 , 4, 4, 4, 4]

rewards = format_and_correctness_reward_func(test_completions, test_answer)

for i, (completion, reward) in enumerate(zip(test_completions, rewards)):
    display_text = completion[:50] + "..." if len(completion) > 50 else completion
    print(f"\n{i+1}. Completion: {display_text}")
    print(f"   Correct Answer: {test_answer[i]}")
    print(f"   Reward Score: {reward:.2f}")
    print(f"   Status: ", end="")

    if reward == 1.0:
        print("✅ Perfect! (Correct format + Correct answer)")
    elif reward == 0.5:
        print("⚠️  Good format but wrong answer")
    elif reward == 0.3:
        print("⚠️  Correct answer but poor format")
    else:
        print("❌ Poor format and wrong answer")


1. Completion: <think> I need to add 2 and 2 together. </think> <...
   Correct Answer: 4
   Reward Score: 1.00
   Status: ✅ Perfect! (Correct format + Correct answer)

2. Completion: <think> Adding 2 and 2 gives us 4. </think> <answe...
   Correct Answer: 4
   Reward Score: 0.50
   Status: ⚠️  Good format but wrong answer

3. Completion: <think> I need to add these numbers. </think> <ans...
   Correct Answer: 4
   Reward Score: 1.00
   Status: ✅ Perfect! (Correct format + Correct answer)

4. Completion: <answer> 4 </answer>
   Correct Answer: 4
   Reward Score: 0.00
   Status: ❌ Poor format and wrong answer

5. Completion: 4
   Correct Answer: 4
   Reward Score: 0.00
   Status: ❌ Poor format and wrong answer


In [ ]:
print("\n" + "="*60)
print("📊 REWARD ANALYSIS:")
print("="*60)

print("""
| Scenario                    | Reward | Meaning                          |
|-----------------------------|--------|----------------------------------|
| ✅ Format OK + Answer Correct | 1.0    | Perfect!                         |
| ✅ Format OK + Answer Wrong  | 0.5    | Format good but wrong answer     |
| ❌ Format Bad + Answer Correct| 0.3    | Correct but poorly formatted     |
| ❌ Format Bad + Answer Wrong | 0.0    | Everything is wrong              |

This reward function encourages the model to:
1. Use the correct format (<think> and <answer> tags)
2. Provide the correct answer
3. This is ideal for training with reinforcement learning!
""")


📊 REWARD ANALYSIS:

| Scenario                    | Reward | Meaning                          |
|-----------------------------|--------|----------------------------------|
| ✅ Format OK + Answer Correct | 1.0    | Perfect!                         |
| ✅ Format OK + Answer Wrong  | 0.5    | Format good but wrong answer     |
| ❌ Format Bad + Answer Correct| 0.3    | Correct but poorly formatted     |
| ❌ Format Bad + Answer Wrong | 0.0    | Everything is wrong              |

This reward function encourages the model to:
1. Use the correct format (<think> and <answer> tags)
2. Provide the correct answer
3. This is ideal for training with reinforcement learning!



In [13]:
from typing_extensions import Pattern
def advanced_reward_func(completions, answer, **kwargs):
  pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
  rewards = []

  for completion, correct_answer in zip(completions, answer):
    match = re.search(pattern, completion, re.DOTALL)

    if match:
      think_content = match.group(1).strip()
      answer_content = match.group(2).strip()
      correct_str = str(correct_answer).strip().lower()

      format_ok = len(think_content) > 20 and len(answer_content) > 0
      answer_clean = answer_content.strip().lower()

      exact_match = (answer_clean == correct_str)

      partial_match = correct_str in answer_clean or answer_clean in correct_str

      if format_ok and exact_match:
        reward = 1.0
      elif format_ok and partial_match:
        reward = 0.7
      elif format_ok and not partial_match:
        reward = 0.5
      elif not format_ok and exact_match:
        reward = 0.3
      else:
        reward = 0.0

      rewards.append(reward)
    else:
      rewards.append(0.0)

  return rewards

test_advanced_completions = [
    "<think> I need to add 2 and 2. </think> <answer> 4 </answer>",                      # Perfect
    "<think> Adding numbers: 2+2. </think> <answer> the answer is 4 </answer>",          # Contains 4
    "<think> Let me calculate 2+2. </think> <answer> 5 </answer>",                       # Wrong
    "<answer> 4 </answer>",                                                               # No think tag
]

test_answers_advanced = [4, 4, 4, 4]

adv_rewards = advanced_reward_func(test_advanced_completions, test_answers_advanced)

print("Advanced Reward Results:")
for i, (comp, reward) in enumerate(zip(test_advanced_completions, adv_rewards)):
    display_text = comp[:40] + "..." if len(comp) > 40 else comp
    print(f"  {i+1}. {display_text}")
    print(f"     → Reward: {reward:.1f}")

Advanced Reward Results:
  1. <think> I need to add 2 and 2. </think> ...
     → Reward: 1.0
  2. <think> Adding numbers: 2+2. </think> <a...
     → Reward: 0.0
  3. <think> Let me calculate 2+2. </think> <...
     → Reward: 0.5
  4. <answer> 4 </answer>
     → Reward: 0.0


In [14]:
print("\n" + "="*60)
print("📚 SUMMARY AND KEY INSIGHTS:")
print("="*60)

summary = """
✅ WHAT YOU LEARNED:
1. How to create a reward function for format checking
2. How to combine format checking with answer correctness
3. Using zip() to iterate over multiple lists
4. Partial scoring for more nuanced rewards

✅ KEY INSIGHTS:
- Reward functions are crucial for reinforcement learning
- Format rewards encourage structured outputs
- Correctness rewards ensure accurate answers
- Combined rewards = Better model behavior

✅ EXERCISE COMPLETED:
- Function checks both format AND correctness
- Returns appropriate scores (0.0, 0.3, 0.5, 1.0)
- Tested with multiple scenarios

✅ USE CASES:
- RLHF (Reinforcement Learning from Human Feedback)
- Instruction tuning
- Fine-tuning models for structured outputs
- Quality assessment of model responses
"""

print(summary)

print("\n🎯 Challenge: You can extend this to:")
print("   - Check for multiple correct answers")
print("   - Use fuzzy matching for partial credit")
print("   - Add penalization for hallucinations")
print("   - Incorporate confidence scores")

print("\n✅ All done! 🚀")


📚 SUMMARY AND KEY INSIGHTS:

✅ WHAT YOU LEARNED:
1. How to create a reward function for format checking
2. How to combine format checking with answer correctness
3. Using zip() to iterate over multiple lists
4. Partial scoring for more nuanced rewards

✅ KEY INSIGHTS:
- Reward functions are crucial for reinforcement learning
- Format rewards encourage structured outputs
- Correctness rewards ensure accurate answers
- Combined rewards = Better model behavior

✅ EXERCISE COMPLETED:
- Function checks both format AND correctness
- Returns appropriate scores (0.0, 0.3, 0.5, 1.0)
- Tested with multiple scenarios

✅ USE CASES:
- RLHF (Reinforcement Learning from Human Feedback)
- Instruction tuning
- Fine-tuning models for structured outputs
- Quality assessment of model responses


🎯 Challenge: You can extend this to:
   - Check for multiple correct answers
   - Use fuzzy matching for partial credit
   - Add penalization for hallucinations
   - Incorporate confidence scores

✅ All done! 🚀
